# AndinaLog 03B | Flota | Diagnóstico v2

Contrato didáctico: datos originales visibles, decisiones trazables y tres salidas CSV.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-FLOTA-diagnostico-didactico-v2"
COLUMNAS_BRONZE = ["camion_id", "centro_distribucion_base", "capacidad_kg", "tipo_camion"]
CENTROS_PERMITIDOS = {"Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"}
TIPOS_PERMITIDOS = {"Seco", "Refrigerado"}
UMBRAL_REVISION_CAPACIDAD_KG = 750

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ = encontrar_raiz()
RUTA_BRONZE = RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_flota.csv"
SALIDAS = RAIZ / "proyecto-integrador/01_diagnostico/andinalog_flota/salidas"
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
if list(bronze.columns) != COLUMNAS_BRONZE:
    raise ValueError(f"Esquema Bronze inesperado: {list(bronze.columns)}")
df = bronze.copy(deep=True)
df.insert(0, "fila_bronze", range(1, len(df) + 1))
print("Filas Bronze:", len(df))


## Aplicación visible de reglas

`*_estado` indica la mayor severidad detectada para el campo y `*_motivo` conserva todas sus explicaciones. Las comprobaciones usan versiones auxiliares del texto, sin sobrescribir el valor Bronze.


In [ ]:
PRIORIDAD = {"OK": 0, "REVISAR": 1, "CRITICO": 2}
for columna in COLUMNAS_BRONZE:
    df[f"{columna}_estado"] = "OK"
    df[f"{columna}_motivo"] = ""

def marcar(columna, mascara, estado, motivo):
    mascara = pd.Series(mascara, index=df.index).fillna(False).astype(bool)
    e, m = f"{columna}_estado", f"{columna}_motivo"
    subir = mascara & df[e].map(PRIORIDAD).lt(PRIORIDAD[estado])
    df.loc[subir, e] = estado
    previo = df.loc[mascara, m]
    df.loc[mascara, m] = previo.where(previo.eq(""), previo + "; ") + motivo

for columna in COLUMNAS_BRONZE:
    marcar(columna, df[columna].str.strip().eq(""), "CRITICO", "Valor faltante")

# Identificador: formato y duplicidad exacta o contradictoria.
id_original = df["camion_id"]
id_limpio = id_original.str.strip()
id_normal = id_limpio.str.upper()
formato_id = id_original.str.fullmatch(r"CAM-\d{2}").fillna(False)
marcar("camion_id", id_limpio.ne("") & ~formato_id, "CRITICO", "Formato esperado: CAM-##")

firma = pd.util.hash_pandas_object(df[COLUMNAS_BRONZE], index=False)
variantes = firma.groupby(id_limpio, dropna=False).transform("nunique")
id_repetido = id_limpio.ne("") & id_limpio.duplicated(keep=False)
conflicto = id_repetido & variantes.gt(1)
copia = df.duplicated(COLUMNAS_BRONZE, keep="first") & ~conflicto
marcar("camion_id", conflicto, "CRITICO", "Mismo ID con datos contradictorios (revisar todas las variantes)")
marcar("camion_id", copia, "CRITICO", "Copia exacta posterior (no contar dos veces)")

# Una variante que colisiona con otro ID válido tras normalizar se revisa; no se fusiona.
colision_normal = (id_normal.ne("") & id_normal.duplicated(keep=False)
                  & ~id_limpio.duplicated(keep=False))
marcar("camion_id", colision_normal, "REVISAR", "Colisión potencial al normalizar espacios o mayúsculas")

centro = df["centro_distribucion_base"].str.strip()
marcar("centro_distribucion_base", centro.ne("") & ~centro.isin(CENTROS_PERMITIDOS),
       "CRITICO", "Centro fuera del catálogo permitido")

capacidad_texto = df["capacidad_kg"].str.strip()
capacidad = pd.to_numeric(capacidad_texto, errors="coerce")
marcar("capacidad_kg", capacidad_texto.ne("") & capacidad.isna(), "CRITICO", "Capacidad no numérica")
marcar("capacidad_kg", capacidad.notna() & capacidad.le(0), "CRITICO", "Capacidad debe ser mayor que 0 kg")
marcar("capacidad_kg", capacidad.gt(0) & capacidad.lt(UMBRAL_REVISION_CAPACIDAD_KG),
       "REVISAR", "Capacidad inferior a 750 kg: verificar ficha técnica del vehículo")

tipo = df["tipo_camion"].str.strip()
marcar("tipo_camion", tipo.ne("") & ~tipo.isin(TIPOS_PERMITIDOS),
       "CRITICO", "Tipo fuera del catálogo: Seco o Refrigerado")

print("Reglas aplicadas; columnas Bronze intactas")


## Resumen por fila y validación

La cuarentena del diagnóstico es un subconjunto del diagnosticado. No se debe concatenar con él al preparar tratamiento.


In [ ]:
# Los estados son auxiliares para decidir la cuarentena; no se exportan.
print("Reglas de Flota evaluadas:",len(df),"filas")


## Exportación

La salida diagnosticada contiene todas las filas y la cuarentena solo las filas críticas. El tratamiento posterior decidirá normalizaciones, exclusiones y cualquier revisión manual de conflictos.


In [ ]:
# Public diagnostic contract: one quarantine flag and one reason per Bronze field.
for campo in COLUMNAS_BRONZE:
    df[f"{campo}_en_cuarentena"] = df[f"{campo}_estado"].eq("CRITICO")
df["en_cuarentena"] = df[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1)
columnas_publicas = ["fila_bronze", *COLUMNAS_BRONZE]
columnas_publicas += [x for c in COLUMNAS_BRONZE for x in (f"{c}_en_cuarentena", f"{c}_motivo")]
columnas_publicas += ["en_cuarentena"]
diagnosticado = df[columnas_publicas].copy()
cuarentena = diagnosticado.loc[diagnosticado["en_cuarentena"]].copy()
conteos = {"filas_bronze":len(bronze), "filas_diagnosticadas":len(diagnosticado),
           "filas_cuarentena":len(cuarentena), "filas_con_observaciones":int(df[[f"{c}_motivo" for c in COLUMNAS_BRONZE]].ne("").any(axis=1).sum())}
for campo in COLUMNAS_BRONZE:
    conteos[f"cuarentena_{campo}"] = int(diagnosticado[f"{campo}_en_cuarentena"].sum())
reporte_calidad = pd.DataFrame([{"metrica":k,"valor":v} for k,v in conteos.items()])
assert len(diagnosticado)==len(bronze)
assert diagnosticado["en_cuarentena"].equals(diagnosticado[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1))
assert len(cuarentena)==int(diagnosticado["en_cuarentena"].sum())
pd.testing.assert_frame_equal(diagnosticado[COLUMNAS_BRONZE],bronze)
SALIDAS.mkdir(parents=True,exist_ok=True)
base="andinalog_flota_didactico_v2_"
diagnosticado.to_csv(SALIDAS/(base+"diagnosticado.csv"),index=False,encoding="utf-8-sig")
cuarentena.to_csv(SALIDAS/(base+"cuarentena.csv"),index=False,encoding="utf-8-sig")
reporte_calidad.to_csv(SALIDAS/(base+"reporte_calidad.csv"),index=False,encoding="utf-8-sig")
print(conteos)
display(diagnosticado.tail(5))
